# Exercise: Molecular Crystal Optimization

In this exercise we will be optimizing the atomic positions and lattice of molecular crystals.  Specifically, we have a structure from the CSD (identifier: COWCAS)

![title](cowcas.png)

and a co-crystal structure for a knoevenagel reaction between barbituric acid and vanillin generated by Clari (https://github.com/the-matter-lab/clari).


In [1]:
from ase.io import read
from ase.visualize import view
from ase.filters import FrechetCellFilter
from ase.optimize import FIRE
from mace.calculators import MACECalculator

cowcas = read('./COWCAS.cif')
my_cocrystal = read('./my_cocrystal.cif')


/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/ase/io/cif.py:411: UserWarning: crystal system 'orthorhombic' is not interpreted for space group Spacegroup(36, setting=1). This may result in wrong setting!
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/ase/spacegroup/spacegroup.py:484: UserWarning: scaled_positions 3 and 13 are equivalent
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/ase/spacegroup/spacegroup.py:484: UserWarning: scaled_positions 5 and 14 are equivalent
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/ase/spacegroup/spacegroup.py:484: UserWarning: scaled_positions 4 and 15 are equivalent
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/ase/spacegroup/spacegroup.py:484: UserWarning: scaled_positions 0 and 16 are equivalent
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-

In [3]:
view(cowcas, viewer='x3d')


In [4]:
view(my_cocrystal, viewer='x3d')

## Some functions to help visualize molecules that go across boundaries


In [5]:
from ase import Atoms
from ase.data import covalent_radii
from ase.neighborlist import NeighborList
from ase.build.surface import add_vacuum
import numpy as np

from collections import deque, defaultdict

def build_bond_graph(atoms: Atoms, 
                     scale: float = 1.15, 
                     skin: float = 0.0,
                     periodic: bool = True
                     ) -> dict[int, list[tuple[int, np.ndarray]]]:
    """
    Build a bonded graph from distances under PBC.

    atoms : ase.Atoms object
    scale : float
        Bond criterion scale factor on covalent radii sum.
    skin : float
        ASE NeighborList skin.

    Returns
    -------
    graph : dict
        graph[i] = list of (j, S_ij)
        where S_ij is the integer lattice shift such that
        r_j + S_ij @ cell - r_i
        is the bonded minimum-image vector.
    """
    numbers = atoms.get_atomic_numbers()
    cell = atoms.cell.array

    # Pair cutoff = scale * (r_cov_i + r_cov_j)
    # NeighborList expects one radius per atom, and pairs interact if overlaps exist
    cutoffs = [scale * covalent_radii[z] for z in numbers]

    nl = NeighborList(cutoffs, skin=skin, bothways=True, self_interaction=False)
    nl.update(atoms)

    graph = defaultdict(list)

    
    for i in range(len(atoms)):
        indices, offsets = nl.get_neighbors(i)

        for j, S in zip(indices, offsets):
            if j <= i:
                continue

            if periodic:
                S = np.asarray(S, dtype=int)
                rij = atoms.positions[j] + S @ cell - atoms.positions[i]
            elif not periodic:
                rij = atoms.positions[j] - atoms.positions[i]

            dij = np.linalg.norm(rij)

            cutoff_ij = scale * (covalent_radii[numbers[i]] + covalent_radii[numbers[j]])

            if dij <= cutoff_ij:
                graph[i].append((j, S))
                graph[j].append((i, -S))

    return graph


def connected_components(graph: dict, 
                         n_atoms: int,
                         ) -> list[list[int]]:
    """
    graph : from build_bond_graph
    Find connected components in the bond graph.
    """
    seen = set()
    components = []

    for start in range(n_atoms):
        if start in seen:
            continue

        comp = []
        queue = deque([start])
        seen.add(start)

        while queue:
            i = queue.popleft()
            comp.append(i)

            for j, _ in graph.get(i, []):
                if j not in seen:
                    seen.add(j)
                    queue.append(j)

        components.append(comp)

    return components


def unwrap_component(atoms: Atoms, 
                     graph: dict, 
                     component: list[int]
                     ) -> tuple[dict[int, np.ndarray], np.ndarray]:
    """
    Assign integer lattice shifts n_i to atoms in one connected component
    so bonded neighbors become contiguous in Cartesian space.

    If atom i has shift n_i, then neighbor j gets:
        n_j = n_i + S_ij
    """
    cell = atoms.cell.array
    component_set = set(component)

    shifts = {}
    root = component[0]
    shifts[root] = np.zeros(3, dtype=int)

    queue = deque([root])

    while queue:
        i = queue.popleft()

        for j, Sij in graph.get(i, []):
            if j not in component_set:
                continue

            proposed = shifts[i] + Sij

            if j not in shifts:
                shifts[j] = proposed
                queue.append(j)
            else:
                # Ignore inconsistency here; usually means overconnected graph
                pass

    for i in component:
        if i not in shifts:
            shifts[i] = np.zeros(3, dtype=int)

    unwrapped = np.array([
        atoms.positions[i] + shifts[i] @ cell
        for i in component
    ])

    return shifts, unwrapped

def repair_fragmented_molecules(
    atoms: Atoms,
    bond_scale: float = 1.15,
    neighbor_skin: float = 0.0,
    centroid_position_frac: np.ndarray = None,
    # return_components=False,
    ) -> Atoms:
    """
    Repair molecules fragmented across periodic boundaries.

    Parameters
    ----------
    atoms : ase.Atoms
    bond_scale : float
        Bond criterion scale factor on covalent radii sum.
    neighbor_skin : float
        ASE NeighborList skin.
    
    Returns
    -------
    repaired : ase.Atoms
        New Atoms object with repaired positions.
    info : dict, optional
        Extra information if return_components=True
    """
    repaired = atoms.copy()
    
    graph = build_bond_graph(repaired, scale=bond_scale, skin=neighbor_skin)
    components = connected_components(graph, len(repaired))

    new_positions = repaired.positions.copy()
    
    for comp_idx, comp in enumerate(components):
        if len(comp) == 1:
            continue

        atom_shifts, unwrapped = unwrap_component(repaired, graph, comp)

        for k, atom_idx in enumerate(comp):
            new_positions[atom_idx] = unwrapped[k]

    repaired.positions[:] = new_positions

    if centroid_position_frac is not None:
        cell = repaired.cell.array
        centroid_cart = np.mean(repaired.positions, axis=0)
        centroid_frac = np.linalg.solve(cell.T, centroid_cart)
        shift_frac = centroid_position_frac - centroid_frac
        shift_cart = shift_frac @ cell
        repaired.positions += shift_cart

    return repaired


In [6]:
view(repair_fragmented_molecules(cowcas),viewer='x3d')

In [7]:
view(repair_fragmented_molecules(my_cocrystal),viewer='x3d')

## Create our MACECalculator object

In [ ]:
# https://mace-docs.readthedocs.io/en/latest/guide/foundation_models.html
#  MODEL is MACE-MP-0 small
MODEL = './2023-12-10-mace-128-L0_energy_epoch-249.model'
calculator = MACECalculator(model_paths=MODEL, 
                            device='cpu',   # 'cuda' for GPU
                            default_dtype='float64')

/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


## Attach our calculator to our ASE objects
Normally when you attach a calculator to a crystal object, the unit cell will not be optimized (such as the Cu(111) examples earlier). In this exercise we will change this so that the atomic positions *and* unit cell are allowed to change during optimization. We do this by applying a filter (FretchetCellFilter) to our object before optimization. Then we use the FIRE algorithm for our optimization.

In [9]:
my_cocrystal.calc = calculator
cowcas.calc = calculator

# Wrapper that applies the Frechet cell filter to the crystals.  Allows cell lattice to relax independently of the atomic positions.
# More info: https://docs.ase-lib.org/ase/filters.html#module-ase.filters
filter_mycocrystal = FrechetCellFilter(my_cocrystal)
filter_cowcas = FrechetCellFilter(cowcas)

opt_my_cocrystal = FIRE(filter_mycocrystal)
opt_cowcas = FIRE(filter_cowcas)


In [10]:
# Let's just get our initial energy 
initial_energy_mycocrystal = my_cocrystal.get_potential_energy()
initial_energy_cowcas = cowcas.get_potential_energy()

print(f"Initial energy of my_cocrystal: {initial_energy_mycocrystal}")
print(f"Initial energy of cowcas: {initial_energy_cowcas}")

Initial energy of my_cocrystal: -419.45219681137735
Initial energy of cowcas: -1076.9561558628143


## Optimize our structures

In [11]:
# Optimize molecular crystals
#   Using the FIRE optimizer to relax the crystal structures with a maximum of 50 steps or to fmax=0.1 eV/angstrom
maxsteps = 50
opt_my_cocrystal.run(fmax=0.1, steps=maxsteps)
opt_cowcas.run(fmax=0.1, steps=maxsteps)


      Step     Time          Energy          fmax
FIRE:    0 18:13:22     -419.452197        3.771344


/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/scipy/_lib/_util.py:1138: RuntimeWarning: logm result may be inaccurate, approximate err = 7.15427717068451e-13
  return f(*arrays, *other_args, **kwargs)


FIRE:    1 18:13:23     -420.171629        1.962920


/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/scipy/_lib/_util.py:1138: RuntimeWarning: logm result may be inaccurate, approximate err = 7.151524094818094e-13
  return f(*arrays, *other_args, **kwargs)


FIRE:    2 18:13:24     -420.091369        2.545257


...

/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/scipy/_lib/_util.py:1138: RuntimeWarning: logm result may be inaccurate, approximate err = 7.162776222599098e-13
  return f(*arrays, *other_args, **kwargs)


FIRE:   49 18:14:09     -420.997511        0.289114


/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/scipy/_lib/_util.py:1138: RuntimeWarning: logm result may be inaccurate, approximate err = 7.215539738737819e-13
  return f(*arrays, *other_args, **kwargs)


FIRE:   50 18:14:10     -421.003334        0.201121
      Step     Time          Energy          fmax
FIRE:    0 18:14:12    -1076.956156        1.601821
FIRE:    1 18:14:14    -1077.594824        0.573285
FIRE:    2 18:14:17    -1077.504649        1.046695
FIRE:    3 18:14:19    -1077.658642        0.831528
FIRE:    4 18:14:21    -1077.835131        0.511228
FIRE:    5 18:14:24    -1077.898676        0.479881
FIRE:    6 18:14:26    -1077.903124        0.447473
FIRE:    7 18:14:28    -1077.911407        0.392528
FIRE:    8 18:14:30    -1077.922483        0.339479
FIRE:    9 18:14:32    -1077.935180        0.275700
FIRE:   10 18:14:35    -1077.948564        0.238423
FIRE:   11 18:14:37    -1077.962225        0.233463
FIRE:   12 18:14:39    -1077.976333        0.227123
FIRE:   13 18:14:41    -1077.993082        0.233739
FIRE:   14 18:14:44    -1078.013610        0.238038
FIRE:   15 18:14:46    -1078.038676        0.226994
FIRE:   16 18:14:48    -1078.067792        0.197185
FIRE:   17 18:

np.True_

In [15]:
view(repair_fragmented_molecules(my_cocrystal), viewer='x3d')


In [16]:
view(repair_fragmented_molecules(cowcas), viewer='x3d')

In [17]:
# Get the energy of the optimized structures
optimized_energy_mycocrystal = my_cocrystal.get_potential_energy()
optimized_energy_cowcas = cowcas.get_potential_energy()

print(f"Optimized energy of my_cocrystal: {optimized_energy_mycocrystal}")
print(f"Optimized energy of cowcas: {optimized_energy_cowcas}")

Optimized energy of my_cocrystal: -421.0033344317584
Optimized energy of cowcas: -1078.223361807823
